# Working with Raw Recordings

In [1]:
import numpy as np
import mne
%matplotlib qt


Raw EEG and MEG data contains recordings from multiple sensors distributed across the scalp. EEG sensors are electrodes that directly pick up the electrical fields generated by neuronal activity. MEG channels come in two types: magnetometers and gradiometers. While magnetometers measure the absolute strength of the magnetic field, gradiometers measure the spatial gradient of the magnetic field between two nearby points. For each MEG sensor location, there are three channels: one magnetometer and two gradiometers. In this notebook, you are going to learn how to load and explore raw EEG and MEG data using MNE-Python.

## Inspecting Raw Data

### Background

In MNE-Python, raw EEG and MEG data is stored in an instance of the `Raw` class. A `Raw` object is a container that stores the sensor values, the time points at which those values were recorded, as well as an `Info` object that contains important metadata such as the names and locations of the sensors, the sampling rate at which the data were recorded and the filters that have been applied to the data. The `Raw` class also provides a `.plot()` method that allows you to explore the data in an interactive graphical interface.

### Exercises

In this section, you are going to learn how to work with `Raw` objects in MNE-Python. You are going to visualize the raw data, read different kinds of metadata from the `Raw` object's `.info` attribute and export the raw sensor recordings as a numpy array. Here are the code examples required to solve the exercises:

| Code | Description |
| --- | --- |
| `mne.datasets.sample.data_path()` | Download the MNE sample data and return the path where it is stored |
| `raw = mne.io.read_raw_fif("recording.fif")` | Read raw data from a FIF file and store it in the variable `raw`|
| `raw.info` | Get the `.info` of the `raw` object |
| `raw.info["ch_names"]` | Get the list of channel names from `raw.info` |
| `raw.info["bads"]` | Get the list of bad channels from `raw.info` |
| `raw.info["sfreq"]` | Get the sampling frequency from `raw.info` |
| `raw.plot()` | Create an interactive plot of the `raw` data |
| `raw.times` | Get the array of `.times` for all samples in `raw` |
| `raw.get_data()` | Get the data array from `raw` |

Execute the cell below to download a sample dataset from MNE and store it in the variable `raw`.

In [2]:
data_path = mne.datasets.sample.data_path()
raw = mne.io.read_raw_fif(data_path/"MEG"/"sample"/"sample_audvis_raw.fif")
raw

**Exercise**: Use the `.plot()` method to visualize the raw data. Use the sidebar to scroll through channels and the left and right arrow keys to move through time. What differences can you spot between MEG and EEG channels?

**Exercise**: Get the `.info` of the `raw` object.

**Example**: Get the sampling frequency (`"sfreq"`) from `raw.info`.

In [5]:
raw.info["sfreq"]

**Exercise**: Get the list of channel names (`"ch_names"`) from `raw.info`.

**Exercise**: Get the list of bad channels (`"bads"`) from `raw.info`.

**Exercise**: Get the array of `.times` from the `raw` object. How many time points have been recorded in this data?

**Exercise**: Get the data array from `raw`. How many channels are there in total?

## Selecting Channels and Data Segments

### Background

MEG and EEG recordings can get very large. The sample dataset we are working with here contains 60 EEG and 306 MEG channels plus some additional auxiliary channels to record the stimuli and electrooculogram (EOG). However, often we want to focus our analysis on a subset of channels where we expect the neural phenomenon of interest. We may also want to select a specific time range, for example when stimuli were presented. In this section, you are going to learn how to pick channels and crop the data to a specific time range. When calling methods such as `.pick()` or `.crop()`, the `Raw` object is modified in-place, which means that the data stored in `Raw` is overwritten with the values returned by those methods. If we want to keep the original data, we can `.copy()` the `Raw` object first and then apply the modifications to the copy.

### Exercises

In this section, you are going to learn to pick channels, either by type or name, from an MNE `Raw` object and crop the data to a specific time range. To keep the original data, we are going to copy the `Raw` object and modify the copy. Don't worry though if you accidentally overwrite the original data! You can simply execute the cell that loads the sample data at the beginning of this notebook to reload the original data. Here are some useful code snippets for solving the exercises in this section:

| Code | Description |
| --- | --- |
| `raw.pick(["eeg"])` | Select all `"eeg"` channels from `raw` |
| `raw.pick(["EEG 017", "EEG 018"])` | Select the channels `"EEG 017"` and `"EEG 018"` from `raw` |
| `raw.crop(tmin=10, tmax=20)` | Crop the data recorded between `10` and `20` seconds |
| `raw2 = raw.copy()` | Create a copy of `raw` and store it in the variable `raw2` |
| `raw.plot_sensors(ch_type="mag")` | Plot the sensor layout for all magnetometers |
| `raw.plot_sensors(ch_type="mag", show_names=True)` | Plot the sensor layout for all magnetometers with their names |

**Example**: Create a copy of `raw` called `raw_eeg` and select all `"eeg"`, `"stim"` and `"eog"` channels.

In [10]:
raw_eeg = raw.copy()
raw_eeg.pick(["eeg", "stim", "eog"])

**Exercise**: Create a copy of `raw` called `raw_meg` and select all `"meg"`, `"stim"` and `"eog"` channels.

**Exercise**: Plot all magnetometer sensors.

**Exercise**: Plot all EEG sensors with their names.

**Example**: Create a copy of `raw` called `raw_eeg_left` and select the channels stored in the list `left`.

In [14]:
left = ["EEG 017", "EEG 018", "EEG 025", "EEG 026"]
raw_eeg_left = raw.copy()
raw_eeg_left.pick(left)

**Exercise**: Create a copy of `raw` called `raw_eeg_right` and select the channels stored in the list `right`.

In [15]:
right = ["EEG 023", "EEG 024", "EEG 034", "EEG 035"]

**Example**: Create a copy of `raw` called `raw_min1` and crop it to the data recorded within the first minute.

In [17]:
raw_min1 = raw.copy()
raw_min1.crop(tmin=0, tmax=60)

**Exercise**: Create a copy of `raw` called `raw_min2` and crop it to the data recorded within the second minute.

## Manual Annotation of Raw EEG Data

### Background

When working with EEG data, it is common to encounter segments that are contaminated by artifacts such as muscle movements, eye blinks, or electrode pops. MNE-Python allows you to mark these segments using annotations. Annotations are metadata that store the onset, duration, and a description for each marked segment. Any annotation whose description starts with `"BAD_"` is treated specially by MNE — these segments will be automatically excluded from further analysis steps like epoching. Annotations can be added interactively through the plotting GUI or programmatically in code. Once segments are annotated, you can extract or remove them using methods like `.crop_by_annotations()`.

### Exercises

In this section, you are going to learn how to inspect, create, and modify annotations on raw data. You will use the interactive GUI to manually mark bad segments and learn how to add annotations programmatically. Finally, there will be a demo that shows how to crop raw data to remove the annotations. Here are the code examples required to solve the exercises:

| Code | Description |
| --- | --- |
| `raw.annotations` | Get the annotations stored in `raw` |
| `raw.annotations[0]` | Get the first annotation |
| `raw.annotations.onset` | Get the onset for all annotations |
| `raw.annotations.description` | Get the description for all annotations |
| `raw.annotations.duration` | Get the duration for all annotations |
| `raw.annotations.append(onset=3, duration=0.5, description="BAD_")` | Add a new annotation to `raw` at `3` seconds with a duration of `0.5` seconds |
| `raw.crop_by_annotations()` | Extract the annotated segments from `raw` as a list of `Raw` objects |
| `mne.concatenate_raws([raw1, raw2])` | Concatenate a list of `Raw` objects into one |

Run the cell below to create a copy of `raw` and select all `"eeg"` channels.

In [19]:
raw_eeg = raw.copy()
raw_eeg.pick("eeg")

**Example**: Print the annotations stored in `raw_eeg`.

In [20]:
raw_eeg.annotations

**Exercise**: Use the `.plot()` method to open the MNE GUI. In the GUI, press **a** to enter annotation mode. Click the "Add new label" button to create a new label with the default name `BAD_`. Then left click and drag to annotate a segment. Annotate at least three data segments, then close the GUI.

**Exercise**: Print the annotations again. It should reflect the number of segments you manually annotated.

**Exercise**: Print the first element in `raw_eeg.annotations` to see the exact onset and duration of the annotation.

**Exercise**: Print the `.onset` for all annotations.

**Exercise**: Print the `.description` for all annotations.

**Example**: Programmatically annotate a segment with the description `"Test"`.

In [27]:
raw_eeg.annotations.append(onset=3, duration=0.5, description="Test")

**Exercise**: Programmatically annotate a segment with a description of your choice.

**Exercise**: Use the `.crop_by_annotations()` method to extract the annotated segments.

**Demo**: Run the cell below to crop the raw data to exclude the segments annotated as `"BAD_"` and create a new `Raw` object called `raw_clean`. Usually, you would not do this explicitly. Instead, MNE will automatically remove `"BAD_"` segments when slicing the continuous raw data into epochs.

In [30]:
onsets = raw_eeg.annotations.onset[raw_eeg.annotations.description == "BAD_"]
durations = raw_eeg.annotations.duration[raw_eeg.annotations.description == "BAD_"]
edges = [raw_eeg.times[0]] + [t for o, d in zip(onsets, durations) for t in (o, o + d)] + [raw_eeg.times[-1]]
pairs = list(zip(edges[::2], edges[1::2]))
raw_clean = mne.concatenate_raws([raw_eeg.copy().crop(tmin=a, tmax=b) for a, b in pairs if b > a])
raw_clean


## Loading Data from Different Device-Formats

### Background

EEG and MEG data can be recorded using devices from different manufacturers, each of which stores data in its own file format. For example, Elekta/NEUROMAG systems use the FIF format, CTF systems store data in a directory with a `.ds` extension, and 4D Neuroimaging/BTI systems use their own binary format. On the EEG side, BrainVision stores data across a triplet of `.vhdr`, `.vmrk`, and `.eeg` files, while the European Data Format (EDF) uses a single `.edf` file. MNE-Python provides dedicated reader functions for each of these formats under `mne.io`, all of which return a `Raw` object with the same interface you have been using throughout this notebook. For a list of all supported file formats, check the [MNE documentation](https://mne.tools/stable/auto_tutorials/io/index.html).

### Exercises

In this section, you are going to load raw data from different file formats. Each exercise provides a path to a sample dataset that will be downloaded automatically to your home directory. Feel free to remove these datasets at the end of the session to free up space on your hard drive. Here are the code examples required to solve the exercises:

| Code | Description |
| --- | --- |
| `mne.io.read_raw_ctf(raw_path)` | Read raw MEG data from the CTF format |
| `mne.io.read_raw_bti(raw_path)` | Read raw MEG data from the 4D Neuroimaging/BTI format |
| `mne.io.read_raw_brainvision(raw_path, preload=True)` | Read raw EEG data from the BrainVision format and preload the data into memory |
| `mne.io.read_raw_edf(raw_path, verbose=False)` | Read raw EEG data from the EDF format without verbose output |

**Exercise**: Read the raw MEG data stored in the CTF format from the `raw_path` defined below and plot it. This dataset contains somatosensory evoked responses to median nerve stimulation recorded with a CTF 275 MEG system.

In [31]:
data_path = mne.datasets.brainstorm.bst_raw.data_path()
raw_path = data_path / "MEG" / "bst_raw" / "subj001_somatosensory_20111109_01_AUX-f.ds"

**Exercise**: Read the raw MEG data stored in the 4D Neuroimaging/BTI format from the `raw_path` defined below and plot it. This dataset contains phantom recordings from a 4D Neuroimaging/BTI MEG system, designed for validating source localization algorithms using a known current source.

In [33]:
data_path = mne.datasets.phantom_4dbti.data_path()
raw_path = data_path / "1" / "e,rfhp1.0Hz"

**Exercise**: Read the raw EEG data stored in the BrainVision format from the `raw_path` defined below with `preload=True` and plot it. This dataset contains 32-channel EEG recordings of a frequency-tagging experiment where participants observed checkerboard patterns inverting at 12 Hz or 15 Hz.

In [35]:
data_path = mne.datasets.ssvep.data_path()
raw_path = data_path / "sub-02" / "ses-01" / "eeg" / "sub-02_ses-01_task-ssvep_eeg.vhdr"


**Exercise**: Read the raw EEG data stored in the EDF format from the `raw_path` defined below with `preload=True` and `verbose=False` and plot it. This dataset contains 64-channel EEG recordings of motor imagery tasks recorded with the BCI2000 system.

In [37]:
raw_path = mne.datasets.eegbci.load_data(subjects=1, runs=6)[0]

**Optional**: Delete the `mne_data/` folder in your home directory to remove the downloaded sample data and free up space.

## DEMO: Creating a RawArray from Scratch

Even though MNE supports a wide variety of file formats, there may be situations where your data can't be imported using the available functions. In this case, it is always possible to create a `RawArray` from scratch. To do this, you just need your data stored as a Numpy array and an `Info` object with the channel names, channel types, and sampling rate (other attributes are optional). Once created, the `RawArray` provides the same interface you have been working with throughout this notebook.
This demo will show how to create a `RawArray` with some randomly generated dummy data and save it in the FIF file format. Here is an explanation of the central code snippets:

| Code | Description |
| --- | --- |
| `np.random.randn(10)` | Generate `10` random numbers |
| `np.random.randn(5, 10)` | Generate a `5`-by-`10` matrix of random numbers |
| `mne.create_info(ch_names=["a", "b"], ch_types="eeg", sfreq=512)` | Generate an info object for two `"eeg"` channels named `"a"` and `"b"` with a sampling frequency of `512` |
| `mne.create_info(ch_names=["a", "b"], ch_types=["mag", "grad"], sfreq=512)` | Generate an info object for two channels where the first is a magnetometer and the second a gradiometer |
| `raw = mne.io.RawArray(data, info)` | Generate a `raw` data object from the `data` with the given `info` |
| `raw.save("recording-raw.fif", overwrite=True)` | Save the `raw` data to a FIF file |

First, we'll generate some dummy data that are just random noise. We'll generate 3 channels each with 1000 samples. Note that MNE always expects data to have the shape `(n_channels, n_samples)`.

In [39]:
n_channels = 3
n_samples = 1000
data = np.random.randn(n_channels, n_samples)
data.shape

Next, we must create an `Info` object for our sensors. Let's assume these are `"eeg"` channels recorded at a sampling frequency of `100` Hz.

In [40]:
info = mne.create_info(ch_names=["CH_1", "CH_2", "CH_3"], ch_types="eeg", sfreq=100)
info

Now we can create a `RawArray` from `data` using the `info` created above. This works exactly like any other `Raw` object in MNE. For example, we can plot it using the MNE GUI.

In [41]:
raw = mne.io.RawArray(data=data, info=info)
raw.plot(scalings="auto");

Finally, we can save the `Raw` object in the FIF file format so future analyses can skip the manual creation of the object. Per MNE's convention, a raw data file always ends in `"-raw.fif"`.

In [42]:
raw.save("dummy-raw.fif", overwrite=True)